# Доп. главы методов оптимизации — домашнее задание № 2

**По материалу занятий 6, 7 и 8:**

* **Занятие 6. Variance reduction.** Задача конечной суммы $\min_x f(x)=\frac1n\sum_{i=1}^n f_i(x)$;
  почему SGD даже для $\mu$-сильно выпуклой $L$-гладкой $f$ сходится лишь сублинейно, $\widetilde O(1/K)$;
  идея оценки $g_k\to\nabla f(x^*)=0$. SAG и SAGA: несмещённость SAGA, функция Ляпунова
  $V_k=\|x_k-x^*\|^2+4n\gamma^2\sigma_k^2$, линейная сходимость с множителем
  $\rho=\max\{1-\gamma\mu,\,1-\tfrac1{2n}\}$, сложность $O\big((n+\tfrac L\mu)\log\tfrac{V_0}\varepsilon\big)$,
  роль инициализации, память $O(nd)$. SVRG (внешний и внутренний циклы, память $O(d)$, полный градиент),
  SARAH (смещённая, но устойчивее), L-SVRG и PAGE, Katyusha с $O\big((n+\sqrt{nL/\mu})\log\tfrac1\varepsilon\big)$;
  единая схема анализа, sketching и SEGA; вопрос о пользе VR на практике (Defazio, 2018).
* **Занятие 7. Распределённая оптимизация и сжатие.** Сервер и $M$ клиентов, узкое место — передача
  от клиента к серверу. Несмещённое сжатие: $\mathbb E\,Q(x)=x$, $\mathbb E\|Q(x)-x\|^2\le(\omega-1)\|x\|^2$;
  Rand-$k$ с $\omega=d/k$, случайное округление. QGD и его окрестность, пропорциональная
  $\sum_m\|\nabla f_m(x^*)\|^2$; DIANA со сдвигами $h_m$ и сложность $O\big((1+\tfrac\omega M)\tfrac L\mu\log\tfrac1\varepsilon\big)$;
  подсчёт в битах. Смещённое сжатие: $\mathbb E\|C(x)-x\|^2\le(1-\tfrac1\beta)\|x\|^2$ (Top-$k$, округление,
  масштабированный знак), расходимость сжатого GD; error feedback, виртуальные итерации, EF21, EF-BV.
* **Занятие 8. Локальные шаги.** Local GD, оценка с членом $\gamma^2L(H-1)^2\sigma_{\rm dif}^2$, где
  $\sigma_{\rm dif}^2=\frac1M\sum_m\|\nabla f_m(x^*)\|^2$; дрейф клиентов и SCAFFOLD. Похожесть данных:
  $\|\nabla^2f_m-\nabla^2f\|\le\delta$, матричное неравенство Хёфдинга и $\delta\sim L/\sqrt N$;
  зеркальный спуск с $\psi=f_1+\frac\delta2\|x\|^2$ и оценка $O\big(\tfrac\delta\mu\log\tfrac1\varepsilon\big)$ раундов;
  DANE, S-DANE, gradient sliding и ускоренный экстраградиент с $O\big(\sqrt{\delta/\mu}\log\tfrac1\varepsilon\big)$.

---

Это домашнее задание: пять экспериментов, каждый проверяет численно одно конкретное
утверждение из лекций.

### Правила

* Ноутбук должен выполняться сверху вниз без правок (`Kernel → Restart & Run All`).
* Пишется **только код методов**. Отрисовка и печать таблиц делаются готовыми функциями
  `plot_curves`, `plot_sweep`, `show_table` из раздела «Общий каркас» — свои варианты
  оформления графиков писать не нужно и не следует.
* Каждый эксперимент заканчивается **письменным выводом** (2–5 предложений). Вывод
  «график построен» — это не вывод; нужно сравнение численного результата с теоретическим
  предсказанием из лекции.
* Все запуски — с фиксированным `seed`; там, где в формуле стоит $\mathbb E$, в коде должно быть
  усреднение по нескольким независимым запускам.
* Все методы внутри одного эксперимента стартуют из **одной и той же** точки $x^0$ — она задана
  в ноутбуке константой; менять её нельзя, иначе кривые несравнимы.
* Разрешены `numpy`, `matplotlib`, `scipy`. Готовые оптимизаторы (`torch.optim`, `sklearn`) использовать
  **нельзя** — все методы реализуются вручную.

### Баллы

| | Задача | Баллы |
|---|---|---|
| Э1 | Variance reduction: SAG, SAGA, SVRG, L-SVRG, SARAH против SGD | 10 |
| Э2 | SAGA: функция Ляпунова, сложность $O\big((n+L/\mu)\log\frac1\varepsilon\big)$, инициализация | 10 |
| Э3 | Несмещённое сжатие: QGD и DIANA, счёт в битах | 10 |
| Э4 | Смещённое сжатие: расходимость, error feedback и EF21 | 10 |
| Э5 | Локальные шаги: Local GD, SCAFFOLD, похожесть данных | 10 |
| | **Итого** | **50** |

### Обозначения

$x^*\in\arg\min f$, $f^*=f(x^*)$, $\|\cdot\|=\|\cdot\|_2$; $\mu$ и $L$ — константы сильной выпуклости и гладкости.
В Э1–Э2 $n$ — число слагаемых $f_i$, «вызов оракула» — одно вычисление $\nabla f_i$, полный градиент стоит $n$ вызовов.
В Э3–Э5 $M$ — число клиентов, $f=\frac1M\sum_m f_m$, $d$ — размерность, «раунд» — одна передача от клиентов
к серверу. Параметр сжатия $\omega\ge1$ понимается как на занятии 7: $\mathbb E\|Q(x)-x\|^2\le(\omega-1)\|x\|^2$,
так что $\omega=1$ — сжатия нет.

## Настройка окружения

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8.0, 4.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "lines.linewidth": 1.9,
    "legend.framealpha": 0.9,
})

SEED = 0
np.set_printoptions(precision=4, suppress=True)
print("numpy", np.__version__)

## Общий каркас

Ниже — утилиты, которые **писать не нужно**: они выдаются готовыми, чтобы у всех совпадали
и формат кода, и вид графиков. Ваша задача — реализовать сами методы; отрисовка и печать таблиц
делаются вызовом этих функций.

**Соглашения, которых нужно придерживаться во всех задачах:**

1. Метод — это функция, которая возвращает **историю** в виде одномерного массива
   (значения $f(x_k)-f^*$, $\|x_k-x^*\|^2$ и т.п.), начиная **с точки $x^0$**: `hist[0]` — значение
   в стартовой точке. Тогда все кривые на графике заведомо выходят из одной точки.
2. Все методы в пределах одного эксперимента стартуют из **одного и того же** $x^0$
   (для каждой задачи он задан в этом ноутбуке как константа).
3. График сходимости строится **только** через `plot_curves`, график «результат против параметра» —
   через `plot_sweep`. В обоих логарифм **только по оси $y$**, ось $x$ линейная; в `plot_sweep`
   параметр отложен в декадах ($\log_{10}$ значения), поэтому узлы сетки не слипаются.
4. Числовые сводки печатаются через `show_table`.

In [ ]:
EPS_PLOT = 1e-16          # пол для логарифмической оси
PALETTE = plt.cm.tab10(np.arange(10))


def best_so_far(v):
    '''min_{i<=k} v_i — «лучшее значение к моменту k».'''
    return np.minimum.accumulate(np.asarray(v, dtype=float))


def plot_curves(curves, xlabel, ylabel, title, ref=None, x=None, ax=None,
                figsize=(8.2, 4.6), legend_fs=9, mark_start=True):
    '''Единый формат графика сходимости: логарифм по y, ЛИНЕЙНАЯ ось x.

    curves : {подпись: 1-D массив}, hist[0] — значение в стартовой точке
    ref    : (подпись, массив) — эталонная кривая, чёрный пунктир
    x      : ось абсцисс (по умолчанию 0, 1, 2, ...)
    '''
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=figsize)
    for (name, y), color in zip(curves.items(), PALETTE):
        y = np.maximum(np.asarray(y, dtype=float), EPS_PLOT)
        ax.plot(np.arange(len(y)) if x is None else x, y, color=color, label=name)
    if ref is not None:
        y = np.maximum(np.asarray(ref[1], dtype=float), EPS_PLOT)
        ax.plot(np.arange(len(y)) if x is None else x, y, "k--", lw=1.2, label=ref[0])
    starts = {float(np.asarray(y, dtype=float)[0]) for y in curves.values()}
    if mark_start and len(starts) == 1:
        x0 = 0 if x is None else np.asarray(x)[0]
        ax.plot([x0], [max(starts.pop(), EPS_PLOT)], "o", color="black", ms=6, zorder=5,
                label="общая стартовая точка")
    ax.set_yscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=legend_fs)
    if own:
        plt.tight_layout(); plt.show()
    return ax


def decades(v):
    '''Параметр -> его десятичный логарифм: ось x у нас всегда ЛИНЕЙНАЯ,
    поэтому параметр, пробегающий порядки величины, откладывается как log10.'''
    return np.log10(np.asarray(v, dtype=float))


def plot_sweep(x, series, xlabel, ylabel, title, hlines=None, vlines=None,
               ax=None, figsize=(8.2, 4.6), legend_fs=9):
    '''График «итог против параметра». Логарифм только по оси y; ось x линейная,
    а сам параметр отложен в декадах (log10), иначе точки сетки сливаются у нуля.

    series : {подпись: массив той же длины, что x}
    hlines : {подпись: уровень}, vlines : {подпись: значение параметра}
    '''
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=figsize)
    for (name, y), color in zip(series.items(), PALETTE):
        ax.plot(decades(x), np.maximum(np.asarray(y, dtype=float), EPS_PLOT),
                "o-", color=color, label=name)
    for name, lvl in (hlines or {}).items():
        ax.axhline(lvl, color="k", ls="--", lw=1.2, label=name)
    for name, pos in (vlines or {}).items():
        ax.axvline(decades(pos), color="grey", ls=":", lw=1.2, label=name)
    ax.set_yscale("log")
    ax.set_xlabel(rf"$\log_{{10}}$ {xlabel}"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=legend_fs)
    if own:
        plt.tight_layout(); plt.show()
    return ax


def show_table(headers, rows, width=15):
    '''Печать сводки в едином формате: float — в экспоненциальной записи.'''
    def cell(v):
        return f"{v:.3e}" if isinstance(v, float) else str(v)
    head = " | ".join(str(h).rjust(width) for h in headers)
    print(head); print("-" * len(head))
    for r in rows:
        print(" | ".join(cell(v).rjust(width) for v in r))

### Задача конечной суммы (Э1, Э2)

$$f(x)=\frac1n\sum_{i=1}^n f_i(x),\qquad f_i(x)=\varphi\big(\langle a_i,x\rangle,\,b_i\big)+\frac\lambda2\|x\|^2,\qquad \|a_i\|=1,$$
где $\varphi(z,b)=\log(1+e^{-bz})$, $b\in\{-1,1\}$ (логистическая регрессия, Э1) или $\varphi(z,b)=\frac12(z-b)^2$
(наименьшие квадраты, Э2). Поскольку $\|a_i\|=1$, **каждое** слагаемое $f_i$ $L$-гладкое: $L=\frac14+\lambda$
у логистической и $L=1+\lambda$ у квадратичной. Именно эта константа стоит в оценках занятия 6. Признаки имеют разный масштаб, поэтому $f$ гораздо глаже отдельных $f_i$: $L_f=\lambda_{\max}(\nabla^2 f)<L$.

In [ ]:
class FiniteSum:
    '''f(x) = (1/n) sum_i f_i(x),  f_i(x) = phi(<a_i, x>, b_i) + lam/2 ||x||^2,  ||a_i|| = 1.'''

    def __init__(self, loss="logistic", n=1000, d=20, lam=1e-3, seed=0):
        rng = np.random.default_rng(seed)
        A = rng.standard_normal((n, d)) * np.logspace(0, -2, d)
        A /= np.linalg.norm(A, axis=1, keepdims=True)
        w = rng.standard_normal(d)
        if loss == "logistic":
            b = np.where(rng.random(n) < 1 / (1 + np.exp(-3 * A @ w)), 1.0, -1.0)
        else:
            b = A @ w + 0.5 * rng.standard_normal(n)
        self.A, self.b, self.lam, self.loss, self.n, self.d = A, b, lam, loss, n, d
        self.L = (0.25 if loss == "logistic" else 1.0) + lam       # гладкость каждого f_i
        x = np.zeros(d)                                             # x* — методом Ньютона
        for _ in range(100):
            x = x - np.linalg.solve(self.hess(x), self.grad(x))
        self.xs, self.fs = x, self.f(x)
        eig = np.linalg.eigvalsh(self.hess(x))
        self.mu, self.Lf = eig[0], eig[-1]                         # для логистической — в точке x*

    def _phi(self, z):
        if self.loss == "logistic":
            return np.logaddexp(0.0, -self.b * z)
        return 0.5 * (z - self.b) ** 2

    def _dphi(self, z, idx=slice(None)):
        b = self.b[idx]
        if self.loss == "logistic":
            return -b * np.exp(-np.logaddexp(0.0, b * z))          # -b * sigmoid(-b z), устойчиво
        return z - b

    def f(self, x):
        return np.mean(self._phi(self.A @ x)) + 0.5 * self.lam * x @ x

    def grad(self, x):
        return self.A.T @ self._dphi(self.A @ x) / self.n + self.lam * x

    def grad_i(self, x, i):
        '''Градиент одного слагаемого: ОДИН вызов оракула.'''
        return self._dphi(self.A[i] @ x, i) * self.A[i] + self.lam * x

    def grads(self, x):
        '''Матрица (n, d) всех градиентов слагаемых: n вызовов оракула.'''
        return self._dphi(self.A @ x)[:, None] * self.A + self.lam * x

    def hess(self, x):
        if self.loss == "logistic":
            s = 1 / (1 + np.exp(-self.b * (self.A @ x)))
            w = s * (1 - s)
        else:
            w = np.ones(self.n)
        return (self.A.T * w) @ self.A / self.n + self.lam * np.eye(self.d)

### Распределённая квадратичная задача (Э3–Э5)

$$f(x)=\frac1M\sum_{m=1}^M f_m(x),\qquad f_m(x)=\frac1{2N}\|A_mx-b_m\|^2+\frac\lambda2\|x\|^2
=\frac12x^{\!\top}H_mx-\langle c_m,x\rangle+\mathrm{const},$$
у $m$-го клиента $N$ своих объектов. Параметр `het` задаёт **неоднородность**: у каждого клиента
свой сдвиг признаков и своя «истинная» модель, поэтому $\nabla f_m(x^*)\neq0$, хотя
$\frac1M\sum_m\nabla f_m(x^*)=0$. Величина $\sigma_{\rm dif}^2=\frac1M\sum_m\|\nabla f_m(x^*)\|^2$ из занятий 7–8
считается в конструкторе. Константы: $\mu=\lambda_{\min}(H)$, $L_f=\lambda_{\max}(H)$ для $H=\frac1M\sum_mH_m$
и $L=\max_m\lambda_{\max}(H_m)$ — гладкость отдельного клиента.

In [ ]:
class Clients:
    '''f(x) = (1/M) sum_m f_m(x),  f_m(x) = 1/2 x^T H_m x - <c_m, x>.'''

    def __init__(self, Hm, cm):
        self.Hm, self.cm = np.asarray(Hm, float), np.asarray(cm, float)
        self.M, self.d = self.cm.shape
        self.H, self.c = self.Hm.mean(0), self.cm.mean(0)
        self.xs = np.linalg.solve(self.H, self.c)
        self.fs = self.f(self.xs)
        eig = np.linalg.eigvalsh(self.H)
        self.mu, self.Lf = eig[0], eig[-1]
        self.L = max(np.linalg.eigvalsh(h)[-1] for h in self.Hm)
        self.gstar = self.grads(self.xs)                       # (M, d): grad f_m(x*)
        self.sigma_dif2 = float(np.mean(np.sum(self.gstar ** 2, axis=1)))

    @classmethod
    def from_data(cls, M=10, N=100, d=50, lam=1e-2, het=1.0, identical=False, seed=0):
        rng = np.random.default_rng(seed)
        scales = np.logspace(0, -1, d)
        x_common = rng.standard_normal(d)
        Hm, cm = [], []
        for m in range(M):
            if m == 0 or not identical:
                A = (rng.standard_normal((N, d)) + het * rng.standard_normal(d)) * scales
                b = A @ (x_common + het * rng.standard_normal(d)) + 0.1 * rng.standard_normal(N)
            Hm.append(A.T @ A / N + lam * np.eye(d))
            cm.append(A.T @ b / N)
        return cls(Hm, cm)

    def grads(self, X):
        '''X: (d,) — общая точка, или (M, d) — у каждого клиента своя. Возвращает (M, d).'''
        X = np.broadcast_to(X, (self.M, self.d))
        return np.einsum("mij,mj->mi", self.Hm, X) - self.cm

    def grad(self, x):
        return self.H @ x - self.c

    def f(self, x):
        return 0.5 * x @ self.H @ x - self.c @ x

    def dist2(self, x):
        return float(np.sum((x - self.xs) ** 2))

---
# Экспериментальные задачи

Общее правило: везде, где в формуле стоит $\mathbb E$, в коде должно стоять усреднение по нескольким
независимым запускам (не по одной траектории!). Все графики — в логарифмическом масштабе по оси $y$.

## Эксперимент Э1 (10 баллов). Variance reduction против SGD

Полигон — логистическая регрессия `FiniteSum("logistic", n=1000, d=20, lam=1e-3)`, старт $x^0=0$.
Шаг у всех методов один и тот же, $\gamma=\frac1{6L}$ — это шаг из теоремы о сходимости SAGA с занятия 6.
Стоимость считается в **вызовах оракула** $\nabla f_i$; «эпоха» — $n$ вызовов.

| Метод | Оценка $g_k$ | Вызовов за итерацию | Что хранит |
|---|---|---|---|
| SGD | $\nabla f_{i_k}(x_k)$ | 1 | — |
| SAG | $\frac1n\sum_i y_i^{k+1}$, где $y_{i_k}^{k+1}=\nabla f_{i_k}(x_k)$, остальные $y_i$ не меняются | 1 | $n$ векторов $y_i$ |
| SAGA | $\nabla f_{i_k}(x_k)-y_{i_k}^k+\frac1n\sum_i y_i^k$, затем $y_{i_k}^{k+1}=\nabla f_{i_k}(x_k)$ | 1 | $n$ векторов $y_i$ |
| SVRG | $\nabla f_{i_k}(x_k)-\nabla f_{i_k}(w)+\nabla f(w)$; каждые $n$ итераций $w\leftarrow x$ | 2, и ещё $n$ при обновлении $w$ | $w$, $\nabla f(w)$ |
| L-SVRG | то же, но $w\leftarrow x_{k+1}$ с вероятностью $p=1/n$ после каждой итерации | 2, и ещё $n$ с вероятностью $p$ | $w$, $\nabla f(w)$ |
| SARAH | в начале внешнего цикла $v=\nabla f(x)$; далее $v_k=\nabla f_{i_k}(x_k)-\nabla f_{i_k}(x_{k-1})+v_{k-1}$; длина цикла $n$ | 2, и ещё $n$ в начале цикла | $x_{k-1}$, $v_{k-1}$ |

**Первоисточники.**

| Метод | Источник |
|---|---|
| SAG | [Le Roux, Schmidt, Bach. *A Stochastic Gradient Method with an Exponential Convergence Rate for Finite Training Sets.* NeurIPS 2012, arXiv:1202.6258](https://arxiv.org/abs/1202.6258); полный анализ — [Schmidt, Le Roux, Bach. *Minimizing Finite Sums with the Stochastic Average Gradient.* Math. Program. 2017, arXiv:1309.2388](https://arxiv.org/abs/1309.2388) |
| SAGA | [Defazio, Bach, Lacoste-Julien. *SAGA: A Fast Incremental Gradient Method With Support for Non-Strongly Convex Composite Objectives.* NeurIPS 2014, arXiv:1407.0202](https://arxiv.org/abs/1407.0202) |
| SVRG | [Johnson, Zhang. *Accelerating Stochastic Gradient Descent using Predictive Variance Reduction.* NeurIPS 2013](https://papers.nips.cc/paper/2013/hash/ac1dd209cbcc5e5d1c6e28598e8cbbe8-Abstract.html) |
| L-SVRG | [Kovalev, Horváth, Richtárik. *Don't Jump Through Hoops and Remove Those Loops: SVRG and Katyusha are Better Without the Outer Loop.* ALT 2020, arXiv:1901.08689](https://arxiv.org/abs/1901.08689) |
| SARAH | [Nguyen, Liu, Scheinberg, Takáč. *SARAH: A Novel Method for Machine Learning Problems Using Stochastic Recursive Gradient.* ICML 2017, arXiv:1703.00102](https://arxiv.org/abs/1703.00102) |
| PAGE | [Li, Bao, Zhang, Richtárik. *PAGE: A Simple and Optimal Probabilistic Gradient Estimator for Nonconvex Optimization.* ICML 2021, arXiv:2008.10898](https://arxiv.org/abs/2008.10898) |
| Katyusha | [Allen-Zhu. *Katyusha: The First Direct Acceleration of Stochastic Gradient Methods.* STOC 2017, JMLR 2018, arXiv:1603.05953](https://arxiv.org/abs/1603.05953) |
| единая схема анализа | [Gorbunov, Hanzely, Richtárik. *A Unified Theory of SGD: Variance Reduction, Sampling, Quantization and Coordinate Descent.* AISTATS 2020, arXiv:1905.11261](https://arxiv.org/abs/1905.11261) |
| SEGA, sketching | [Hanzely, Mishchenko, Richtárik. *SEGA: Variance Reduction via Gradient Sketching.* NeurIPS 2018, arXiv:1809.03054](https://arxiv.org/abs/1809.03054) |
| польза VR в глубоком обучении | [Defazio, Bottou. *On the Ineffectiveness of Variance Reduced Optimization for Deep Learning.* NeurIPS 2019, arXiv:1812.04529](https://arxiv.org/abs/1812.04529) |
| обзор | [Gower, Schmidt, Bach, Richtárik. *Variance-Reduced Methods for Machine Learning.* Proc. IEEE 108(11), 2020, arXiv:2010.00892](https://arxiv.org/abs/2010.00892) |

**Формат кода.** Каждый метод — класс-наследник `Estimator` (заготовка ниже). Общий цикл `run_vr` дан: он
выбирает $i_k$, делает шаг $x_{k+1}=x_k-\gamma g_k$, считает вызовы оракула и раз в эпоху записывает
$f(x_k)-f^*$. Метод `candidates` возвращает $g_k$ для **каждого** возможного $i_k$ при текущем состоянии;
по нему цикл считает точное (без Монте-Карло) условное смещение $\|\mathbb E_{i_k}g_k-\nabla f(x_k)\|/\|\nabla f(x_k)\|$
и условную ошибку $\mathbb E_{i_k}\|g_k-\nabla f(x_k)\|^2$.

**Что нужно сделать.**

1. Реализовать шесть методов из таблицы.
2. Построить $f(x_k)-f^*$ по эпохам (30 эпох, усреднение по 3 запускам) и вывести таблицу
   «метод | $f-f^*$ после 30 эпох | множитель за эпоху», множитель — $(\text{gap}_{30}/\text{gap}_{10})^{1/20}$.
3. Запустить с `diag=True` (один запуск) и построить $\mathbb E_{i_k}\|g_k-\nabla f(x_k)\|^2$ по эпохам. Добавить уровень
   $\sigma_*^2=\frac1n\sum_i\|\nabla f_i(x^*)\|^2$ — к нему должна стремиться ошибка SGD.
   Вывести таблицу относительного смещения на эпохах 1, 5, 10: какие оценки несмещённые, какие нет?
4. Посчитать, сколько чисел хранит каждый метод при $n=1000$, $d=20$, и ответить: как для обобщённой
   линейной модели $f_i(x)=\varphi(\langle a_i,x\rangle)+\frac\lambda2\|x\|^2$ уменьшить память SAGA до $O(n)$?
5. **Вывод:** почему SGD с тем же шагом выходит на плато, а методы с VR — нет (свяжите с графиком ошибки
   оценки); кто быстрее в пересчёте на вызовы оракула; откуда «ступеньки» у SVRG; мешает ли смещение
   SAG и SARAH сходиться.

In [ ]:
class Estimator:
    '''Оценка g_k ~ grad f(x_k). Всё, что метод помнит между итерациями (таблица y_i,
    опорная точка w, предыдущая точка), хранится в самом объекте.'''
    name = "base"

    def start(self, prob, x0, rng):
        '''Подготовка в стартовой точке. Вернуть число вызовов оракула, потраченных на неё.'''
        self.prob = prob
        return 0

    def step(self, x, i):
        '''Индекс i = i_k уже выбран. Вернуть (g_k, число вызовов оракула) и обновить таблицы.'''
        raise NotImplementedError

    def after_step(self, x_new, rng):
        '''Вызывается после x_{k+1} = x_k - gamma * g_k: здесь SVRG, L-SVRG и SARAH
        обновляют опорную точку. Вернуть число потраченных вызовов оракула.'''
        return 0

    def candidates(self, x):
        '''(n, d): чему было бы равно g_k при КАЖДОМ возможном i_k в текущем состоянии.
        Состояние не меняет и в счёт вызовов не идёт — нужно только для диагностики.'''
        raise NotImplementedError


def run_vr(est, prob, x0, gamma, epochs, seed=0, diag=False):
    '''Общий цикл Э1. Отсчёт — каждые n вызовов оракула (эпоха); hist[0] — в точке x0.
    Возвращает словарь массивов: "gap" = f(x) - f*, а при diag=True ещё
    "bias" = ||E_i g - grad f|| / ||grad f||  и  "mse" = E_i ||g - grad f||^2,
    усреднённые по всем итерациям эпохи (в нулевой точке — значение в x0).'''
    rng = np.random.default_rng(seed)
    x = np.array(x0, dtype=float)
    out = {"gap": [], "bias": [], "mse": []}
    acc = {"bias": [], "mse": []}
    calls = est.start(prob, x, rng)

    def measure():
        g_true = prob.grad(x)
        G = est.candidates(x)
        acc["bias"].append(np.linalg.norm(G.mean(0) - g_true) / np.linalg.norm(g_true))
        acc["mse"].append(np.mean(np.sum((G - g_true) ** 2, axis=1)))

    def record():
        out["gap"].append(prob.f(x) - prob.fs)
        if diag:
            for key in acc:                  # за эпоху могло не быть ни одной итерации
                out[key].append(np.mean(acc[key]) if acc[key] else out[key][-1])
                acc[key].clear()

    if diag:
        measure()
    record()
    while len(out["gap"]) <= epochs:
        if diag:
            measure()
        g, c = est.step(x, rng.integers(prob.n))
        x = x - gamma * g
        calls += c + est.after_step(x, rng)
        while calls >= len(out["gap"]) * prob.n and len(out["gap"]) <= epochs:
            record()
    return {k: np.array(v) for k, v in out.items()}


PROB1 = FiniteSum("logistic", n=1000, d=20, lam=1e-3, seed=SEED)
X0_1 = np.zeros(PROB1.d)                       # ОДНА стартовая точка для всего Э1
GAMMA1 = 1 / (6 * PROB1.L)
print(f"n={PROB1.n}, d={PROB1.d}, L={PROB1.L:.4f}, L_f={PROB1.Lf:.4f}, mu(x*)={PROB1.mu:.2e}, gamma=1/(6L)={GAMMA1:.3f}")

In [ ]:
# TODO: классы SGD, SAG, SAGA, SVRG, LSVRG, SARAH — наследники Estimator (методы start / step / after_step / candidates)
# TODO: прогоны run_vr (3 запуска, 30 эпох), график f - f*, таблица множителей за эпоху
# TODO: diag=True — график E||g_k - grad f(x_k)||^2 с уровнем sigma_*^2, таблица смещения; таблица памяти

> **Вывод Э1.** …

## Эксперимент Э2 (10 баллов). SAGA: функция Ляпунова, сложность и инициализация

Полигон — наименьшие квадраты `FiniteSum("square", ...)`: для них $x^*$, $\mu$ и $L$ известны точно.
На занятии 6 для SAGA с шагом $\gamma\le\frac1{6L}$ доказано
$$\mathbb E\,V_{k+1}\le\rho\,\mathbb E\,V_k,\qquad V_k=\|x_k-x^*\|^2+4n\gamma^2\sigma_k^2,\qquad
\sigma_k^2=\frac1n\sum_{i=1}^n\|y_i^k-\nabla f_i(x^*)\|^2,\qquad \rho=\max\Big\{1-\gamma\mu,\ 1-\frac1{2n}\Big\},$$
откуда $\mathbb E\,V_K\le\rho^KV_0$ и $K=O\big((n+\frac L\mu)\log\frac{V_0}\varepsilon\big)$. У градиентного спуска
каждая итерация стоит $n$ вызовов, и вызовов нужно $O\big(n\frac{L_f}\mu\log\frac1\varepsilon\big)$.

**Что нужно сделать.** Шаг SAGA везде $\gamma=\frac1{6L}$, старт $x^0=0$.

**(а) Функция Ляпунова.** Реализовать SAGA, возвращающую историю $V_k$ и $\|x_k-x^*\|^2$
(величину $\sigma_k^2$ пересчитывайте за $O(d)$ на итерации, меняется только одно слагаемое).
Для $n=500$ рассмотреть два режима: $\lambda=10^{-1}$ (тогда $\frac L\mu\approx10$ и $\rho=1-\frac1{2n}$) и
$\lambda=10^{-3}$ ($\frac L\mu\approx10^3$, $\rho=1-\gamma\mu$). Усреднить $V_k$ по 20 запускам, построить его вместе
с $\rho^kV_0$, оценить фактический множитель за итерацию по второй половине траектории и сравнить с $\rho$.
Какой из двух членов в $\max\{\cdot\}$ на самом деле ограничивает скорость в каждом режиме?

**(б) Инициализация** («очень важна инициализация» — замечание с занятия). Сравнить
$y_i^0=\nabla f_i(x^0)$ (стоит $n$ вызовов), $y_i^0=0$ (бесплатно) и $y_i^0=\nabla f_i(x^*)$ (на практике недоступно,
нужно как эталон). Вывести таблицу «инициализация | $V_0$ | вызовов до $\|x-x^*\|^2\le10^{-8}\|x^0-x^*\|^2$»
(вызовы на инициализацию тоже считаются). Объяснить результат через вид $V_0$.

**(в) Сложность.** Для $n\in\{300,1000,3000\}$ и $\lambda\in\{10^{-1},10^{-2},10^{-3},10^{-4}\}$ посчитать число
вызовов оракула до $\|x-x^*\|^2\le10^{-8}\|x^0-x^*\|^2$ для SAGA и для GD с шагом $1/L_f$. Построить их против
$\kappa=L/\mu$ через `plot_sweep` и вывести таблицу с нормировкой: вызовы SAGA$\,/\,(n+\kappa)$ и
вызовы GD$\,/\,(n\kappa_f)$, где $\kappa_f=L_f/\mu$. Если оценки из лекции точны по порядку, нормированные
числа почти не меняются, хотя $n$ и $\kappa$ меняются на порядки. Добавить столбец с нормировкой на то, что даёт
сама теорема: $K\approx\frac{\ln(1/\varepsilon)}{1-\rho}=\max\{6\kappa,\,2n\}\ln\frac1\varepsilon$ при $\gamma=\frac1{6L}$.
Отдельно посчитать, во сколько раз по теории выиграла бы Katyusha, у которой полная оценка
$O\big((n+\sqrt{n\kappa})\log\frac1\varepsilon\big)$: отношение $(n+\kappa)/(n+\sqrt{n\kappa})$ в узлах сетки.

**Вывод:** выполняется ли $\mathbb E V_k\le\rho^kV_0$ и насколько оценка точна; где проходит излом
«$n$ против $\kappa$»; во сколько раз SAGA выигрывает у GD и почему выигрыш растёт с $\kappa$; когда ускорение
Katyusha заметно, а когда нет.

In [ ]:
X0_2 = np.zeros(20)                            # ОДНА стартовая точка для всего Э2


def gd_calls(prob, x0, tol=1e-8):
    '''Вызовов оракула у GD с шагом 1/L_f до ||x - x*||^2 <= tol ||x0 - x*||^2 (итерация = n вызовов).'''
    x = np.array(x0, dtype=float)
    H = prob.hess(x)                           # для квадратичной задачи гессиан постоянный
    g_lin = prob.A.T @ prob.b / prob.n
    e0, k = np.sum((x - prob.xs) ** 2), 0
    while np.sum((x - prob.xs) ** 2) > tol * e0:
        x = x - (H @ x - g_lin) / prob.Lf
        k += 1
    return k * prob.n

In [ ]:
def saga_ls(prob, x0, gamma, K, init="grad", seed=0, tol=None):
    '''SAGA для FiniteSum("square").

    init : "grad" — y_i^0 = grad f_i(x0) (n вызовов), "zero" — y_i^0 = 0, "opt" — y_i^0 = grad f_i(x*)
    tol  : если задан, остановиться, как только ||x_k - x*||^2 <= tol * ||x0 - x*||^2
    return: (V, D, calls) — истории V_k и ||x_k - x*||^2 (длины <= K+1) и общее число вызовов оракула
    '''
    # TODO
    raise NotImplementedError

# TODO (а): два режима lambda, 20 запусков, E V_k против rho^k V_0, фактический множитель
# TODO (б): таблица по трём инициализациям
# TODO (в): сетка n x lambda, вызовы SAGA и GD, plot_sweep против kappa, нормированная таблица, Katyusha

> **Вывод Э2.** …

## Эксперимент Э3 (10 баллов). Несмещённое сжатие: QGD и DIANA

Полигон — `Clients.from_data(M=10, N=200, d=100, lam=0.1, het=0.5)`, старт $x^0=0$, метрика — $\|x_k-x^*\|^2$.
Клиенты считают градиенты точно; узкое место — передача от клиента к серверу, поэтому стоимость
считается в **битах, переданных одним клиентом**. Вещественное число — 32 бита.

**Компрессоры** (применяются к каждой строке матрицы $(M,d)$, у каждого клиента своя случайность):

* **Rand-$k$:** $Q(x)=\frac dk\sum_{i\in S}[x]_ie_i$, $S$ — случайное подмножество из $k$ координат; $\omega=d/k$.
  Передаются $k$ значений и $k$ номеров: $k\,(32+\lceil\log_2d\rceil)$ бит.
* **Случайное округление к степени двойки** («natural compression»): если $2^e\le|x_i|<2^{e+1}$, то
  $|x_i|$ заменяется на $2^{e+1}$ с вероятностью $(|x_i|-2^e)/2^e$ и на $2^e$ иначе, знак сохраняется.
  Это несмещённый компрессор с $\omega=9/8$; передаются знак и порядок — 9 бит на координату.

**Методы** (занятие 7):

* **QGD:** $x_{k+1}=x_k-\gamma\frac1M\sum_mQ(\nabla f_m(x_k))$. Для $\mu$-сильно выпуклых $L$-гладких $f_m$ на лекции получено
  $$\mathbb E\|x_{k+1}-x^*\|^2\le(1-\gamma\mu)\,\mathbb E\|x_k-x^*\|^2+\frac{2\gamma^2\omega}{M^2}\sum_{m=1}^M\|\nabla f_m(x^*)\|^2 ,$$
  то есть сходимость лишь в окрестность радиуса $\frac{2\gamma\omega}{\mu M}\sigma_{\rm dif}^2$: даже в точке $x^*$
  сжатые градиенты клиентов шумят, потому что $\nabla f_m(x^*)\ne0$.
* **DIANA:** клиент хранит сдвиг $h_m^k$ (вначале $0$) и сжимает **разность** $\Delta_m^k=\nabla f_m(x_k)-h_m^k$:
  $$\hat\Delta_m^k=Q(\Delta_m^k),\qquad \hat g_m^k=h_m^k+\hat\Delta_m^k,\qquad h_m^{k+1}=h_m^k+\alpha\hat\Delta_m^k,\qquad
  x_{k+1}=x_k-\gamma\frac1M\sum_m\hat g_m^k .$$
  Сервер знает все $h_m^k$ (он видит те же $\hat\Delta_m^k$), поэтому передаётся только $\hat\Delta_m^k$.
  При $\alpha\omega\le1$ сдвиги $h_m^k\to\nabla f_m(x^*)$, шум сжатия исчезает, и сходимость линейная:
  $K=O\big((1+\frac\omega M)\frac L\mu\log\frac1\varepsilon\big)$ итераций.

**Первоисточники.**

| Тема | Источник |
|---|---|
| квантование, QSGD | [Alistarh, Grubic, Li, Tomioka, Vojnovic. *QSGD: Communication-Efficient SGD via Gradient Quantization and Encoding.* NeurIPS 2017, arXiv:1610.02132](https://arxiv.org/abs/1610.02132) |
| разреживание, Rand-$k$ | [Wangni, Wang, Liu, Zhang. *Gradient Sparsification for Communication-Efficient Distributed Optimization.* NeurIPS 2018, arXiv:1710.09854](https://arxiv.org/abs/1710.09854) |
| округление к степени двойки | [Horváth, Ho, Horváth, Sahu, Canini, Richtárik. *Natural Compression for Distributed Deep Learning.* MSML 2022, arXiv:1905.10988](https://arxiv.org/abs/1905.10988) |
| DIANA | [Mishchenko, Gorbunov, Takáč, Richtárik. *Distributed Learning with Compressed Gradient Differences.* arXiv:1901.09269, 2019](https://arxiv.org/abs/1901.09269); [Horváth, Kovalev, Mishchenko, Stich, Richtárik. *Stochastic Distributed Learning with Gradient Quantization and Variance Reduction.* Optim. Methods Softw. 2023, arXiv:1904.05115](https://arxiv.org/abs/1904.05115) |
| единая теория со сжатием | [Gorbunov, Hanzely, Richtárik. *A Unified Theory of SGD.* AISTATS 2020, arXiv:1905.11261](https://arxiv.org/abs/1905.11261) |

**Что нужно сделать.**

**(а) Компрессоры.** Реализовать оба. На векторе $x\sim\mathcal N(0,I_d)$ по $10^4$ повторам оценить
относительное смещение $\|\overline{Q(x)}-x\|/\|x\|$ (черта — среднее по повторам; для несмещённого
компрессора оно убывает как $1/\sqrt{\text{число повторов}}$) и отношение $\overline{\|Q(x)-x\|^2}/\|x\|^2$, сравнить его с
$\omega-1$. Таблица: «компрессор | смещение | $\overline{\|Q(x)-x\|^2}/\|x\|^2$ | $\omega-1$ | бит на вектор».

**(б) QGD.** Для Rand-$k$ с $k\in\{5,20\}$ и шагов $\gamma_0=\frac1{L(1+2\omega/M)}$, $\gamma_0/4$ построить
$\|x_k-x^*\|^2$ (среднее по 5 запускам). Измерить уровень плато (медиана по последней четверти итераций)
и вывести таблицу «$k$ | $\gamma$ | плато | граница $\frac{2\gamma\omega}{\mu M}\sigma_{\rm dif}^2$ | отношение».
Проверить, что плато пропорционально $\gamma$. Повторить для **одинаковых** клиентов
(`identical=True`, тогда $\sigma_{\rm dif}^2=0$): есть ли плато?

**(в) DIANA.** $\alpha=1/\omega$, $\gamma=\frac1{L(1+2\omega/M)}$. Сравнить GD без сжатия, QGD и DIANA с Rand-5,
DIANA со случайным округлением — по **битам на клиента** (`at_budget` ниже переводит историю по итерациям
в историю по битам). Проверить, что $\frac1M\sum_m\|h_m^k-\nabla f_m(x^*)\|^2\to0$.

**(г) Биты против $\omega$.** Для $M\in\{2,8,32\}$ (первые $M$ клиентов задачи с $M=32$) и
$k\in\{100,50,20,10,5,2,1\}$ найти число бит на клиента до $\|x-x^*\|^2\le10^{-8}\|x^0-x^*\|^2$ у DIANA с
Rand-$k$ ($\alpha$, $\gamma$ как в (в)). Построить против $\omega=d/k$ через `plot_sweep`. По теории
бит нужно $\propto k\,(1+\frac\omega M)\propto d\,(\frac1\omega+\frac1M)$ (на лекции — запись «$\omega\ge M$»):
вывести таблицу отношения измеренного к $d\,(\frac1\omega+\frac1M)$.

**Вывод:** откуда у QGD плато и от чего оно зависит; что именно исправляют сдвиги DIANA; до какого
$\omega$ сжимать выгодно и почему дальше выигрыша в битах нет; во сколько раз DIANA при $\omega\approx M$
экономит биты по сравнению с GD без сжатия.

In [ ]:
PROB3 = Clients.from_data(M=10, N=200, d=100, lam=0.1, het=0.5, seed=SEED)
X0_3 = np.zeros(PROB3.d)                       # ОДНА стартовая точка для Э3 и Э4
BITS_IDX = int(np.ceil(np.log2(PROB3.d)))       # бит на номер координаты


def bits_dense(d):
    return 32 * d


def bits_sparse(k):
    return k * (32 + BITS_IDX)


def bits_natural(d):
    return 9 * d


def at_budget(hist, bits_per_round, budget):
    '''Значение истории после того, как клиент передал не больше budget бит.
    hist[r] — после r раундов, раунд стоит bits_per_round бит; где истории не хватило — nan.'''
    r = np.floor(np.asarray(budget, dtype=float) / bits_per_round).astype(int)
    out = np.full(len(r), np.nan)
    ok = r < len(hist)
    out[ok] = np.asarray(hist)[r[ok]]
    return out


def first_below(hist, tol):
    '''Первый номер k, для которого hist[k] <= tol * hist[0]; nan, если такого нет.'''
    hit = np.flatnonzero(np.asarray(hist) <= tol * hist[0])
    return hit[0] if len(hit) else np.nan


print(f"M={PROB3.M}, d={PROB3.d}, mu={PROB3.mu:.3f}, L={PROB3.L:.2f}, L/mu={PROB3.L / PROB3.mu:.0f}, "
      f"sigma_dif^2={PROB3.sigma_dif2:.3f}")

In [ ]:
# Компрессор — функция C(G, rng): G имеет форму (M, d), сжатие по строкам, результат той же формы.
# Метод — функция (prob, C, x0, gamma, ...) -> история ||x_k - x*||^2 длины K+1.

# TODO: rand_k(G, k, rng), natural(G, rng); проверка смещения и omega, таблица
# TODO: qgd(prob, C, x0, gamma, K, seed); плато против gamma и omega; одинаковые клиенты
# TODO: diana(prob, C, x0, gamma, alpha, K, seed, return_h=False); сравнение по битам; h_m -> grad f_m(x*)
# TODO: сетка M x k, биты до 1e-8, plot_sweep против omega, таблица отношения к d(1/omega + 1/M)

> **Вывод Э3.** …

## Эксперимент Э4 (10 баллов). Смещённое сжатие: расходимость, error feedback и EF21

**Смещённые компрессоры** (занятие 7): $\mathbb E\|C(x)-x\|^2\le(1-\frac1\beta)\|x\|^2$, $\beta\ge1$, но $\mathbb E\,C(x)\ne x$.

* **Top-$k$:** оставить $k$ наибольших по модулю координат; $\beta=d/k$ в худшем случае.
* **Масштабированный знак:** $C(x)=\frac{\|x\|_1}d\operatorname{sign}(x)$; $\beta=d$ в худшем случае; передаётся $d$ бит знака и одно число.

**Методы.**

* **Сжатый GD (CGD):** $x_{k+1}=x_k-\gamma\frac1M\sum_mC(\nabla f_m(x_k))$ — тот же QGD, но со смещённым $C$.
* **EF** (error feedback): клиент отправляет $g_m^k=C(\nabla f_m(x_k)+e_m^k)$ и запоминает то, что не отправил:
  $e_m^{k+1}=e_m^k+\nabla f_m(x_k)-g_m^k$, $e_m^0=0$; сервер делает $x_{k+1}=x_k-\gamma\frac1M\sum_mg_m^k$.
  Анализ на лекции идёт через **виртуальные итерации** $\hat x_k=x_k-\gamma\frac1M\sum_me_m^k$, для которых
  $\hat x_{k+1}=\hat x_k-\gamma\nabla f(x_k)$ — обычный GD, но с градиентом в «чужой» точке.
* **EF21:** $g_m^0=C(\nabla f_m(x^0))$, далее $x_{k+1}=x_k-\gamma\frac1M\sum_mg_m^k$,
  $g_m^{k+1}=g_m^k+C\big(\nabla f_m(x_{k+1})-g_m^k\big)$. Это DIANA с $\alpha=1$, в которой несмещённый $Q$
  заменён на смещённый $C$ (замечание с лекции «DIANA + $\alpha=1$ = EF21»).

**Контрпример** (Безносиков и др.): $M=3$, $d=3$, $f_m(x)=\langle a_m,x\rangle^2+\frac\mu2\|x\|^2$,
$a_1=(-3,2,2)$, $a_2=(2,-3,2)$, $a_3=(2,2,-3)$. Функция $f$ сильно выпукла, $x^*=0$.

**Первоисточники.**

| Тема | Источник |
|---|---|
| смещённое сжатие, контрпример | [Beznosikov, Horváth, Richtárik, Safaryan. *On Biased Compression for Distributed Learning.* JMLR 24, 2023, arXiv:2002.12410](https://arxiv.org/abs/2002.12410) |
| error feedback, 1-bit SGD (Microsoft, 2014) | [Seide, Fu, Droppo, Li, Yu. *1-Bit Stochastic Gradient Descent and its Application to Data-Parallel Distributed Training of Speech DNNs.* Interspeech 2014](https://www.isca-archive.org/interspeech_2014/seide14_interspeech.html) |
| анализ EF, один узел | [Stich, Cordonnier, Jaggi. *Sparsified SGD with Memory.* NeurIPS 2018, arXiv:1809.07599](https://arxiv.org/abs/1809.07599); [Stich, Karimireddy. *The Error-Feedback Framework.* arXiv:1909.05350](https://arxiv.org/abs/1909.05350) |
| EF и знак | [Karimireddy, Rebjock, Stich, Jaggi. *Error Feedback Fixes SignSGD and other Gradient Compression Schemes.* ICML 2019, arXiv:1901.09847](https://arxiv.org/abs/1901.09847) |
| EF21 | [Richtárik, Sokolov, Fatkhullin. *EF21: A New, Simpler, Theoretically Better, and Practically Faster Error Feedback.* NeurIPS 2021, arXiv:2106.05203](https://arxiv.org/abs/2106.05203) |
| EF-BV | [Condat, Yi, Richtárik. *EF-BV: A Unified Theory of Error Feedback and Variance Reduction Mechanisms for Biased and Unbiased Compression in Distributed Optimization.* NeurIPS 2022, arXiv:2205.04180](https://arxiv.org/abs/2205.04180) |

**Что нужно сделать.**

**(а) Компрессоры.** Реализовать Top-$k$ и масштабированный знак. На $10^3$ случайных векторах
$x\sim\mathcal N(0,I_d)$, $d=100$, найти среднее и худшее $\|C(x)-x\|^2/\|x\|^2$ и сравнить с $1-\frac1\beta$
из худшего случая. Таблица «компрессор | среднее | худшее | $1-1/\beta$ | бит на вектор».

**(б) Контрпример.** Вывести (на бумаге), что из точки $x^0=t\,(1,1,1)$ сжатый GD с Top-1 даёт
$x^1=t\big(1+\frac{\gamma(6-\mu)}3\big)(1,1,1)$, то есть расходится при **любом** $\gamma>0$. Проверить численно:
CGD с Top-1 для $\gamma\in\{10^{-3},10^{-2},5\cdot10^{-2}\}$ ($\mu=0.1$, $x^0=(1,1,1)$) — измеренный множитель за
итерацию против формулы; EF и EF21 с теми же $\gamma$ — сходятся ли?

**(в) Сравнение по битам.** На задаче из Э3 при $k=5$ сравнить CGD (Top-5), EF (Top-5), EF21 (Top-5),
EF21 (масштабированный знак), DIANA (Rand-5, $\alpha=k/d$) и GD без сжатия. Шаги из теории у этих методов
очень разные и сильно перестраховываются, поэтому сравнение честное только после подбора: для каждого
метода взять из сетки $\gamma=2^{j}/L$, $j=-8,\dots,5$, шаг, при котором меньше всего бит уходит на достижение
$\|x-x^*\|^2\le10^{-10}\|x^0-x^*\|^2$ (если точность не достигается при бюджете 2 Мбит — шаг с наименьшим
$\|x-x^*\|^2$ в конце бюджета; лучший шаг не должен оказаться на краю сетки). Построить лучшие кривые против
бит на клиента и вывести таблицу «метод | лучший $\gamma L$ | бит до $10^{-10}$ | $\|x-x^*\|^2$ при 2 Мбит».

**(г) Виртуальные итерации.** Для EF (с лучшим шагом из (в)) проверить численно тождество
$\hat x_{k+1}=\hat x_k-\gamma\nabla f(x_k)$ (максимальное отклонение по траектории) и построить
$\bar e_k=\frac1M\sum_me_m^k$, $\frac1M\sum_m\|e_m^k\|$ и $\|x_k-x^*\|^2$ по итерациям. Выходит ли EF на плато?
Измерить плато для шагов $\gamma,\ \gamma/2,\ \gamma/4$ (60 000 итераций, медиана по последним 10 000)
и сравнить его с $\gamma^2\|\bar e\|^2$. Объяснить через $x_k-\hat x_k=\gamma\bar e_k$, почему накопленная ошибка
у неоднородных клиентов **не** уходит в ноль и почему у EF21 такой проблемы нет.

**Вывод:** почему смещённый компрессор ломает сжатый GD даже без всякой стохастики; что чинит EF и
что дополнительно чинит EF21; что выгоднее при одинаковом числе бит — несмещённый Rand-$k$ с DIANA
или смещённый Top-$k$ с EF21.

In [ ]:
def make_counterexample(mu=0.1):
    '''Пример Безносикова и др.: f_m(x) = <a_m, x>^2 + mu/2 ||x||^2, x* = 0.'''
    a = np.array([[-3.0, 2.0, 2.0], [2.0, -3.0, 2.0], [2.0, 2.0, -3.0]])
    Hm = [2 * np.outer(am, am) + mu * np.eye(3) for am in a]
    return Clients(Hm, np.zeros((3, 3)))


PROB4 = make_counterexample(mu=0.1)
X0_4 = np.ones(3)                              # стартовая точка контрпримера
print(f"контрпример: lambda_min(H)={PROB4.mu:.3f}, L={PROB4.L:.1f}, x*={PROB4.xs}")

In [ ]:
# TODO: top_k(G, k, rng=None), scaled_sign(G, rng=None); таблица средних и худших ||C(x)-x||^2/||x||^2
# TODO: ef(prob, C, x0, gamma, K, seed, track=False), ef21(prob, C, x0, gamma, K, seed); CGD — это qgd из Э3
# TODO: контрпример: множитель за итерацию против 1 + gamma(6 - mu)/3; EF и EF21
# TODO: сравнение по битам на задаче Э3; проверка виртуальных итераций

> **Вывод Э4.** …

## Эксперимент Э5 (10 баллов). Локальные шаги: Local GD, SCAFFOLD, похожесть данных

По материалу занятия 8. Напомним схемы.

* **Local GD.** В раунде $k$ каждый клиент стартует из $x^k$ и делает $H$ шагов градиентного спуска по своей
  функции: $x^k_{m,l+1}=x^k_{m,l}-\gamma\nabla f_m(x^k_{m,l})$, $l=0,\dots,H-1$; сервер усредняет
  $x^{k+1}=\frac1M\sum_mx^k_{m,H}$. Один раунд — одна передача. При точных градиентах оценка с лекции
  $$f(\tilde x_T)-f^*=O\Big(\frac{R_0^2}{\gamma T}+\gamma^2L(H-1)^2\sigma_{\rm dif}^2\Big),\qquad T=KH,\qquad
  \sigma_{\rm dif}^2=\frac1M\sum_m\|\nabla f_m(x^*)\|^2 ,$$
  то есть локальные шаги ускоряют начало, но оставляют плато: клиент «уезжает» к своему $x_m^*$ (дрейф клиентов).
  При $H=1$ это обычный GD.
* **SCAFFOLD.** Клиент хранит поправку $c_m$, сервер — $c$ (вначале нули). Локальные шаги
  $x^k_{m,l+1}=x^k_{m,l}-\gamma\big(\nabla f_m(x^k_{m,l})-c_m+c\big)$; после них
  $c_m^+=c_m-c+\frac1{H\gamma}(x^k-x^k_{m,H})$; клиент отправляет $(x^k_{m,H}-x^k,\ c_m^+-c_m)$; сервер:
  $x^{k+1}=\frac1M\sum_mx^k_{m,H}$, $c\leftarrow c+\frac1M\sum_m(c_m^+-c_m)$; клиент: $c_m\leftarrow c_m^+$.
* **Похожесть данных.** Если $\|\nabla^2f_m(x)-\nabla^2f(x)\|\le\delta$, а данные у клиентов из одного распределения,
  то по матричному неравенству Хёфдинга $\delta\sim L/\sqrt N$, где $N$ — объём данных клиента.
  Первый клиент играет роль сервера и делает зеркальный спуск с $\psi=f_1+\frac\delta2\|x\|^2$:
  $$x^{k+1}=\arg\min_x\Big\{\langle\nabla(f-f_1)(x^k),x\rangle+f_1(x)+\frac\delta2\|x-x^k\|^2\Big\}.$$
  Из $\nabla^2\psi\succeq\nabla^2f\succeq(1+\frac{2\delta}\mu)^{-1}\nabla^2\psi$ получается $O\big(\frac\delta\mu\log\frac1\varepsilon\big)$
  раундов вместо $O\big(\frac L\mu\log\frac1\varepsilon\big)$ у GD. Для квадратичных $f_m$ подзадача решается явно:
  $x^{k+1}=x^k-(\nabla^2f_1+\delta I)^{-1}\nabla f(x^k)$.

**Первоисточники.**

| Тема | Источник |
|---|---|
| Local GD, оценка с $\sigma_{\rm dif}^2$ | [Khaled, Mishchenko, Richtárik. *Tighter Theory for Local SGD on Identical and Heterogeneous Data.* AISTATS 2020, arXiv:1909.04746](https://arxiv.org/abs/1909.04746) |
| FedAvg | [McMahan, Moore, Ramage, Hampson, Agüera y Arcas. *Communication-Efficient Learning of Deep Networks from Decentralized Data.* AISTATS 2017, arXiv:1602.05629](https://arxiv.org/abs/1602.05629) |
| SCAFFOLD | [Karimireddy, Kale, Mohri, Reddi, Stich, Suresh. *SCAFFOLD: Stochastic Controlled Averaging for Federated Learning.* ICML 2020, arXiv:1910.06378](https://arxiv.org/abs/1910.06378) |
| SCAFFOLD при похожести гессианов | [Luo, Stich, Horváth, Takáč. *Revisiting LocalSGD and SCAFFOLD: Improved Rates and Missing Analysis.* AISTATS 2025, arXiv:2501.04443](https://arxiv.org/abs/2501.04443) |
| матричное неравенство Хёфдинга | [Tropp. *User-Friendly Tail Bounds for Sums of Random Matrices.* Found. Comput. Math. 12, 2012, arXiv:1004.4389](https://arxiv.org/abs/1004.4389) |
| DANE | [Shamir, Srebro, Zhang. *Communication-Efficient Distributed Optimization using an Approximate Newton-type Method.* ICML 2014, arXiv:1312.7853](https://arxiv.org/abs/1312.7853) |
| зеркальный спуск с $\psi=f_1+\frac\delta2\|x\|^2$ | [Hendrikx, Xiao, Bubeck, Bach, Massoulié. *Statistically Preconditioned Accelerated Gradient Method for Distributed Optimization.* ICML 2020, arXiv:2002.10726](https://arxiv.org/abs/2002.10726) |
| S-DANE | [Jiang, Rodomanov, Stich. *Stabilized Proximal-Point Methods for Federated Optimization.* NeurIPS 2024, arXiv:2407.07084](https://arxiv.org/abs/2407.07084) |
| gradient sliding | [Lan. *Gradient Sliding for Composite Optimization.* Math. Program. 159, 2016, arXiv:1406.0919](https://arxiv.org/abs/1406.0919) |
| ускоренный экстраградиент, $O(\sqrt{\delta/\mu}\log\frac1\varepsilon)$ | [Kovalev, Beznosikov, Borodich, Gasnikov, Scutari. *Optimal Gradient Sliding and its Application to Distributed Optimization Under Similarity.* NeurIPS 2022, arXiv:2205.15136](https://arxiv.org/abs/2205.15136) |
| сжатие и похожесть вместе | [Bylinkin, Beznosikov. *Accelerated Methods with Compressed Communications for Distributed Optimization Problems under Data Similarity.* AAAI 2025, arXiv:2412.16414](https://arxiv.org/abs/2412.16414) |

**Что нужно сделать.** Метрика — $f(x^k)-f^*$ по **раундам связи**, старт $x^0=0$.

**(а) Local GD и плато.** На задаче `PROB5` (неоднородные клиенты) при $\gamma_0=\frac1{4L}$ построить
$f(x^k)-f^*$ для $H\in\{1,2,4,8,16\}$ — должна получиться картинка с лекции. Измерить плато (значение после
4000 раундов) для $\gamma\in\{\gamma_0,\gamma_0/2,\gamma_0/4\}$ и $H\in\{2,4,8,16,32\}$ и вывести таблицу отношения
плато к $\gamma^2(H-1)^2L\sigma_{\rm dif}^2$. При каких $\gamma H$ отношение почти постоянно, а при каких перестаёт
быть постоянным, и почему (подсказка: куда сходятся локальные шаги при $H\to\infty$)? Проверить на одинаковых
клиентах (`identical=True`), что плато исчезает.

**(б) SCAFFOLD.** Реализовать, построить те же кривые, вывести таблицу «$H$ | раундов до $10^{-10}$ у SCAFFOLD |
плато Local GD».

**(в) Похожесть данных.** Однородные клиенты (`het=0`): $M=10$, $d=20$, $\lambda=10^{-3}$, $N\in\{50,200,10^3,5\cdot10^3,2\cdot10^4\}$.
1. Посчитать $\delta(N)=\max_m\|\nabla^2f_m-\nabla^2f\|_2$ и построить против $N$ вместе с $L/\sqrt N$.
2. Для каждого $N$ найти число раундов до $f-f^*\le10^{-8}(f(x^0)-f^*)$ у GD с шагом $1/L_f$ и у метода
   с $\psi=f_1+\frac\delta2\|x\|^2$; построить против $N$ и сравнить с $1+\frac{2\delta}\mu$ и $\frac{L_f}\mu$.
3. Что будет, если взять в $\psi$ вместо $\delta$ величину $0,\ \delta/8,\dots,4\delta$ (для $N=50$ и $N=10^3$)?
   Для квадратичной задачи итерация линейна: $x^{k+1}-x^*=(I-(\nabla^2f_1+\delta'I)^{-1}\nabla^2f)(x^k-x^*)$.
   Сравнить число раундов с условием $\nabla^2\psi\succeq\nabla^2f$ из теории и со спектральным радиусом этой
   матрицы. Достаточно ли условие из теории? Необходимо ли?

**Вывод:** чем платят за локальные шаги и от чего зависит плато; как SCAFFOLD его убирает; почему при похожих
данных число раундов падает с ростом $N$ и что на это тратит клиент-сервер.

In [ ]:
PROB5 = Clients.from_data(M=10, N=200, d=20, lam=0.05, het=0.5, seed=SEED)
PROB5_same = Clients.from_data(M=10, N=200, d=20, lam=0.05, het=0.5, identical=True, seed=SEED)
X0_5 = np.zeros(PROB5.d)                       # ОДНА стартовая точка для всего Э5
print(f"M={PROB5.M}, d={PROB5.d}, mu={PROB5.mu:.3f}, L={PROB5.L:.2f}, L/mu={PROB5.L / PROB5.mu:.0f}, "
      f"sigma_dif^2={PROB5.sigma_dif2:.3f}")


def gap(prob, x):
    return prob.f(x) - prob.fs

In [ ]:
# TODO (а): local_gd(prob, x0, gamma, H, R) -> история f(x^k) - f* длины R+1; картинка по H; таблица плато
# TODO (б): scaffold(prob, x0, gamma, H, R) -> история f(x^k) - f*; раунды до 1e-10
# TODO (в): delta(prob); precond_gd(prob, x0, delta, R) по формуле x - (H_1 + delta I)^{-1} grad f(x);
#           раунды против N; неверное delta

> **Вывод Э5.** …

---

### Что стоит проверить перед сдачей

1. Ноутбук выполняется целиком (`Kernel → Restart & Run All`) без ошибок.
2. В каждом эксперименте есть письменный вывод, и он **сопоставляет** численный результат
   с теоретической оценкой из лекции: Э1 и Э2 — с оценками для SAGA, SVRG и SARAH (занятие 6),
   Э3 и Э4 — с оценками для QGD, DIANA, EF и EF21 (занятие 7), Э5 — с оценками для Local GD,
   SCAFFOLD и методов, использующих похожесть данных (занятие 8).
3. Графики построены **только** через `plot_curves` / `plot_sweep`: у всех кривых внутри одного
   эксперимента одна стартовая точка, подписаны оси и легенда.
4. Числовые сводки выведены через `show_table`.
5. Все запуски воспроизводимы (зафиксированы `seed`).